# Workflow 1: Flat React Agent

In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
from typing import List, Dict, Any

import uuid
import operator
from typing import TypedDict, Literal, Optional, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from configs.setting import settings
from configs.GetConfig import config

from src.LLMService import LLMService
from src.e_agents.guardrail_call import GuardrailCall
from src.e_agents.rejection_call import RejectionCall

from src.d_tools import (
    product_search, 
    product_compare,
    policy_search,
    order_lookup,
)

from src.d_tools import (
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
)

from src.f_prompts import (
    FULL_MASTER_PROMPT,        
    FULL_REJECTION_PROMPT
)

from app.core.security import verify_supabase_jwt


In [2]:
available_tools = {
    "product_search": product_search,
    "product_compare": product_compare,
    "policy_search": policy_search,
    "order_lookup": order_lookup
}

tools_schema = [
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
]


In [3]:
class MasterAgent:
    def __init__(self, llm_service: LLMService, config):
        self.llm_service = llm_service
        self.model = config.llm.google.available[0]  

        # Danh sách tool cần authentication - chỉ cần tên tool
        # Thêm tool mới: chỉ append vào list, KHÔNG sửa logic invoke()
        self.AUTH_TOOLS = ["order_lookup", "cart_lookup", "wishlist_update"]

    def invoke(
        self, 
        messages: List[Dict[str, Any]],
        available_tools: Dict[str, Any] = None,
        tools_schema: List[Dict[str, Any]] = None,
        auth_context: dict = None
        ):
        """
        auth_context (dict, optional): Chứa thông tin xác thực để inject vào tools cần auth
            - user_id: UUID của user đã xác thực (verify từ JWT token)
            - user_token: JWT access token gốc (để tạo Supabase client với RLS)
        """

        start_time = time.time()
        first_token_time = None
        total_input_tokens = 0
        total_output_tokens = 0
        # Lưu lịch sử tool calls (args + output) để master_node inject vào lượt sau
        tool_context = []

        def _sanitize_tool_args(name, args):
            """Chuẩn hóa và chỉ giữ các key hợp lệ trong tool args để debug/dùng lại."""

            def _try_parse_object_string(s):
                """Thử parse chuỗi dạng { ... } thành dict. Không bắt buộc PyYAML."""
                if not isinstance(s, str):
                    return None
                s = s.strip()
                if len(s) < 2 or not (s[0] == '{' and s[-1] == '}'):
                    return None
                # 1. JSON chuẩn
                try:
                    import json
                    parsed = json.loads(s)
                    if isinstance(parsed, dict):
                        return parsed
                except Exception:
                    pass
                # 2. YAML nếu có
                try:
                    import yaml
                    parsed = yaml.safe_load(s)
                    if isinstance(parsed, dict):
                        return parsed
                except Exception:
                    pass
                # 3. Fallback: thêm dấu ngoặc kép cho key không có dấu ngoặc, rồi json.loads
                try:
                    import json
                    import re as _re
                    normalized = _re.sub(r'([a-zA-Z_]\w*)\s*:', r'"\1":', s)
                    parsed = json.loads(normalized)
                    if isinstance(parsed, dict):
                        return parsed
                except Exception:
                    pass
                return None

            # ====================================================================================================================

            if name == "product_search":
                allowed_query_keys = {"keyword", "brand", "category", "min_price", "max_price", "name_contains", "mode", "limit", "include_details", "need_price_info"}

                # Args top-level có thể là string object, list queries, hoặc dict
                if isinstance(args, str):
                    parsed = _try_parse_object_string(args)
                    if isinstance(parsed, dict):
                        args = parsed
                    else:
                        args = {"queries": [args]}
                elif isinstance(args, list):
                    args = {"queries": args}
                elif not isinstance(args, dict):
                    args = {}

                # Giá trị mặc định từ top-level (để merge vào các query thiếu)
                top_defaults = {k: args[k] for k in allowed_query_keys if k in args}
                top_limit = args.get("limit")
                default_limit = top_limit if top_limit is not None else 3

                # Nếu LLM gọi dạng flat (không có queries, keyword ở top-level) hoặc dùng 'query'
                if "queries" not in args:
                    if "query" in args:
                        args = {"queries": [args["query"]]}
                    elif "keyword" in args:
                        args = {"queries": [top_defaults]}
                    else:
                        args = {"queries": []}

                queries = args.get("queries") or []
                if isinstance(queries, str):
                    queries = [queries]

                clean_queries = []
                if isinstance(queries, list):
                    for q in queries:
                        # Nếu q là string object, parse trước
                        if isinstance(q, str):
                            parsed_q = _try_parse_object_string(q)
                            if isinstance(parsed_q, dict):
                                q = parsed_q
                            else:
                                q = {"keyword": q}

                        if isinstance(q, dict):
                            # Nếu keyword là object-string (model copy JSON vào keyword), parse và merge
                            merged = top_defaults.copy()
                            kw = q.get("keyword")
                            parsed_kw = _try_parse_object_string(kw) if isinstance(kw, str) else None
                            if isinstance(parsed_kw, dict):
                                merged.update(parsed_kw)
                                for k, v in q.items():
                                    if k != "keyword":
                                        merged[k] = v
                            else:
                                merged.update(q)

                            if not merged.get("keyword"):
                                merged["keyword"] = f"{merged.get('brand','')} {merged.get('category','')}".strip() or "sản phẩm"
                            
                            # Tránh name_contains quá ngắn/gây nhiễu (vd chỉ 'S')
                            nc = merged.get("name_contains")
                            if nc is not None and len(str(nc).strip()) <= 2:
                                merged["name_contains"] = merged.get("keyword")
                            
                            if "limit" not in merged or merged.get("limit") is None:
                                if merged.get("mode") == "lines":
                                    merged["limit"] = 30
                                else:
                                    merged["limit"] = default_limit
                            clean = {k: v for k, v in merged.items() if k in allowed_query_keys}
                            clean_queries.append(clean)

                result = {"queries": clean_queries}
                if top_limit is not None:
                    result["limit"] = top_limit
                return result

            if name == "product_compare":
                if not isinstance(args, dict):
                    args = {}
                product_names = args.get("product_names") or args.get("products") or args.get("product_name")
                if isinstance(product_names, str):
                    product_names = [product_names]
                if not isinstance(product_names, list):
                    product_names = []
                return {"product_names": [p for p in product_names if isinstance(p, str)]}

            if name == "policy_search":
                if not isinstance(args, dict):
                    args = {}
                return {k: v for k, v in args.items() if k in {"key_word", "limit"}}

            if name == "order_lookup":
                if not isinstance(args, dict):
                    args = {}
                return {k: v for k, v in args.items() if k in {"order_id"}}

            return args if isinstance(args, dict) else {}

        # ====================================================================================================================

        max_turns = config.agent.max_turns
        for turn in range(max_turns):
            turn_start_time = time.time()
            first_token_time = None
            turn_input_tokens = 0
            turn_output_tokens = 0

            # ============================================================
            print(f"🌀 --- LƯỢT {turn + 1} (STREAMING) ---")    
            # ============================================================

            # Gọi API Gemini với streaming — LUÔN True
            response_stream = self.llm_service.call_gemini(
                model=self.model,
                messages=messages,
                tools=tools_schema,
                stream=True
            )
            
            text_content = ""
            fn_accum = {}          # idx-part -> {"name","args","thought_signature"}
            fn_order = []
            is_tool_turn = None
            first_token_time = None

            # Parse STREAMING response: đọc từng chunk
            for chunk in response_stream:
                if first_token_time is None:
                    first_token_time = time.time() - turn_start_time

                    # ============================================================
                    print(f"⏱️ [TTFT] Chữ/Tool đầu tiên xuất hiện sau: {first_token_time:.2f}s\n")
                    # ============================================================
                if getattr(chunk, "usage_metadata", None):
                    if chunk.usage_metadata.prompt_token_count:
                        turn_input_tokens = chunk.usage_metadata.prompt_token_count
                    if chunk.usage_metadata.candidates_token_count:
                        turn_output_tokens = chunk.usage_metadata.candidates_token_count

                if not (chunk.candidates and chunk.candidates[0].content and chunk.candidates[0].content.parts):
                    continue

                parts = chunk.candidates[0].content.parts

                if is_tool_turn is None:
                    is_tool_turn = any(getattr(p, "function_call", None) for p in parts)

                for idx, p in enumerate(parts):
                    fc = getattr(p, "function_call", None)
                    sig = getattr(p, "thought_signature", None)

                    if fc and fc.name:
                        if idx not in fn_accum:
                            fn_accum[idx] = {"name": fc.name, "args": fc.args, "thought_signature": None}
                            fn_order.append(idx)
                        if sig:
                            fn_accum[idx]["thought_signature"] = sig
                    elif getattr(p, "text", None):
                        text_content += p.text
                        # Chỉ in ra ngoài khi KHÔNG phải tool-call turn
                        if not is_tool_turn:
                            print(p.text, end="", flush=True)
                    elif sig and fn_order:
                        # Signature "mồ côi" đến ở part riêng -> gán bù cho function_call gần nhất
                        last_idx = fn_order[-1]
                        if fn_accum[last_idx]["thought_signature"] is None:
                            fn_accum[last_idx]["thought_signature"] = sig

            # Build tool_calls_dict SAU KHI đã đọc hết stream của turn
            tool_calls_dict = {}
            for i, idx in enumerate(fn_order):
                v = fn_accum[idx]
                tool_calls_dict[i] = {
                    "id": f"call_gemini_{turn}_{i}",
                    "name": v["name"],
                    "arguments": json.dumps(v["args"]) if isinstance(v["args"], dict) else str(v["args"]),
                    "thought_signature": v["thought_signature"]
                }

            if first_token_time is None:
                first_token_time = time.time() - turn_start_time

            turn_elapsed = time.time() - turn_start_time
            total_input_tokens += turn_input_tokens
            total_output_tokens += turn_output_tokens
            
            # ============================================================
            print(f"\n\n⏱️ [TURN {turn + 1} LATENCY]: {turn_elapsed:.2f}s") 
            print(f"📊 [TURN {turn + 1} TOKENS]: Input = {turn_input_tokens} | Output = {turn_output_tokens} | Subtotal = {turn_input_tokens + turn_output_tokens}")
            # ============================================================


            # Format lại tool_calls thành cấu trúc chuẩn OpenAI để lưu history
            formatted_tool_calls = []
            for idx, tc in tool_calls_dict.items():
                item = {
                    "id": tc["id"],
                    "type": "function",
                    "function": {
                        "name": tc["name"],
                        "arguments": tc["arguments"]
                    }
                }
                if tc.get("thought_signature"):
                    item["thought_signature"] = tc["thought_signature"]
                formatted_tool_calls.append(item)

            # Lưu message của assistant vào history
            agent_msg = {
                "role": "assistant",
                "content": text_content if text_content else None
            }
            if formatted_tool_calls:
                agent_msg["tool_calls"] = formatted_tool_calls
                
            messages.append(agent_msg)
            
            # Thực thi Tools nếu LLM yêu cầu
            if formatted_tool_calls:

                # ============================================================
                print("\n🔧 LLM yêu cầu gọi Tool...")
                # ============================================================
                
                tool_start_time = time.time()
                
                for tc in formatted_tool_calls:
                    func_name = tc["function"]["name"]
                    func_args = json.loads(tc["function"]["arguments"]) if tc["function"]["arguments"] else {}
                    func_args = _sanitize_tool_args(func_name, func_args)
                    
                    # Inject auth cho tool trong danh sách AUTH_TOOLS
                    if func_name in self.AUTH_TOOLS and auth_context:
                        func_args["current_user_id"] = auth_context.get("user_id")
                        func_args["user_token"] = auth_context.get("user_token")
                        # ============================================================
                        print(f"   🔑 Injected auth for {func_name}: user_id={auth_context.get('user_id')}, user_token={'***' if auth_context.get('user_token') else None}")
                        # ============================================================

                    if func_name in available_tools:
                        real_function = available_tools[func_name]
                        # ============================================================
                        print(f"   👉 Chạy hàm: {func_name}({func_args})")
                        # ============================================================

                        result = real_function(**func_args)
                        # ============================================================
                        print(f"   📊 Kết quả từ Tool: {result}")
                        # ============================================================

                        # Lưu lại args + output vào tool_context để truyền sang lượt sau
                        tool_context.append({
                            "tool": func_name,
                            "args": _sanitize_tool_args(func_name, func_args),
                            "output": str(result),
                        })

                        messages.append({
                            "role": "tool",
                            "tool_call_id": tc["id"],
                            "name": func_name,
                            "content": str(result)
                        })
                    else:
                        # ============================================================
                        print(f"   ❌ Lỗi: Không tìm thấy tool '{func_name}' trong available_tools!")
                        # ============================================================

                tool_elapsed = time.time() - tool_start_time
                # ============================================================
                print(f"⏱️ [TOOL EXECUTION TIME]: {tool_elapsed:.2f}s")
                print("🔄 Gửi kết quả Tool lại cho Gemini suy luận tiếp...\n")
                # ============================================================

                # Nghỉ 1s trước khi sang lượt mới
                time.sleep(1)
                continue
            else:
                total_elapsed = time.time() - start_time
                # ============================================================
                print("\n==================================================")
                print(f"✅ --- HOÀN THÀNH HOÀN TOÀN ---")
                print(f"⏱️ [TOTAL AGENT LATENCY]: {total_elapsed:.2f}s")
                print(f"📊 [TOTAL AGENT TOKENS]: Input = {total_input_tokens} | Output = {total_output_tokens} | Grand Total = {total_input_tokens + total_output_tokens}")
                print(f"TPM: {(total_input_tokens + total_output_tokens) * 60 / total_elapsed}")
                print("==================================================\n")
                # ============================================================
                
                return {
                    "content": text_content if text_content else None,
                    "tool_context": tool_context,
                    "latency": total_elapsed,
                    "tokens": {
                        "input": total_input_tokens,
                        "output": total_output_tokens,
                    }
                }

In [4]:
llm_service = LLMService(settings, config)
guardrail_call = GuardrailCall(llm_service, config)
rejection_call = RejectionCall(llm_service, config)
master_agent = MasterAgent(llm_service, config)

In [5]:
class RetrievedChunk(TypedDict):
    content: str
    source: str
    score: float
    chunk_type: str

class AgentState(TypedDict):

    # 1. INput user
    user_query: str
    session_id: str

    # 2. Auth
    user_token: Optional[str]
    user_id: Optional[str]
    is_authenticated: bool

    # 3. Guardrail & Quality
    risk_level: Optional[str]               # "low" | "medium" | "high"
    relevance_score: float                   # Dùng cho self-check Corrective RAG

    # 4. Router (Định tuyến)
    intent: str                              # Kết quả phân loại: 'product', 'policy', 'account', 'support'
    selected_agent: Optional[str]            # Quyết định Nút xử lý tiếp theo

    # 5. Retrieval & Tools
    retrieved_context: list[RetrievedChunk]
    tool_calls_used: Annotated[list[dict], operator.add]  # ⚡ Reducer cộng dồn lịch sử tool calls (mỗi dict = {tool, args, response?})
    iteration_count: int                     # Đếm số lần lặp chống infinite loop

    # 6. Hội thoại
    messages: Annotated[list, add_messages]  # ⚡ Reducer cộng dồn tin nhắn
    conversation_state: dict

    # 7. Output
    final_answer: Optional[str]
    cited_sources: list[str]
    ticket_id: Optional[str]
    show_popup: bool

    # 8. Thống kê
    input_tokens: Annotated[int, operator.add]
    output_tokens: Annotated[int, operator.add]
    latency: Annotated[float, operator.add]
    total_tokens: Annotated[int, operator.add]

In [6]:
def receive_node(state: AgentState) -> dict:
    query = state.get("user_query", "").strip()
    token = state.get("user_token")
    user_id = None

    if token:
        user_id = verify_supabase_jwt(token)
    
    is_authenticated = True if user_id else False

    # ============================================================
    # if is_authenticated:
    #     print(f"Người dùng đã xác thực")
    # else:
    #     print(f"Người dùng chưa xác thực")
    # ============================================================
    
    return {
        "user_id": user_id,
        "is_authenticated": is_authenticated,
        "user_query": query  
    }

In [7]:
def guardrail_node(state: AgentState) -> dict:
    query = state["user_query"]
    
    result = guardrail_call.invoke(query)
    risk_level = result["risk_level"]
    
    # Chỉ lưu tin nhắn khi KHÔNG phải attack
    if risk_level != "attack":
        return {
            "risk_level": risk_level,
            "show_popup": False,
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ],
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }
    else:
        # Attack → KHÔNG lưu
        return {
            "risk_level": risk_level,
            "show_popup": True,
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }

In [8]:
def rejection_node(state: AgentState) -> dict:
    """
    [NODE] Rejection Agent (Từ chối):
    - Chỉ chạy khi risk_level == "needs_ticket"
    - Trả về câu từ chối lịch sự
    - KHÔNG lưu tin nhắn vào messages (đã lưu ở guardrail_node)
    """
    
    query = state["user_query"]

    result = rejection_call.invoke(query)

    return {
        "final_answer": result["content"],
        "show_popup": True,
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }

In [9]:
def master_node(state: AgentState) -> dict:
    """
    [NODE 3] Master Agent (Tư duy):
    - Chỉ chạy khi risk_level != "attack"
    - Build messages: system prompt + lịch sử tool calls được inject giữa các lượt hội thoại
      (mỗi lượt user sau lượt 1 sẽ nhìn thấy tool/tham số đã gọi ở lượt trước)
    - messages chỉ lưu user query và assistant final answer
    - tool_calls_used: list[dict] lưu lịch sử tool calls theo thứ tự lượt hội thoại
    """
    import json

    def _fmt_search_context(record):
        """Tóm tắt bộ lọc product_search đã dùng ở lượt trước, nhấn mạnh tái sử dụng."""
        if not record or record.get("tool") != "product_search":
            return ""
        args = record.get("args", {})
        queries = args.get("queries", [])
        if not queries:
            return ""
        q = queries[0]
        out = str(record.get("output", ""))
        n_items = 0
        if "Dòng:" in out:
            n_items = out.count("Dòng:")
        elif "Product:" in out:
            n_items = out.count("Product:")
        summary = f"{n_items} dòng" if "Dòng:" in out else (f"{n_items} sản phẩm" if "Product:" in out else "kết quả")
        ctx = (
            f"[BỐI CẢNH TRA CỨU LƯỢT TRƯỚC]\n"
            f"Đã gọi product_search với: {json.dumps(q, ensure_ascii=False)}\n"
            f"Kết quả: {summary}.\n"
            f"QUAN TRỌNG: Nếu câu hỏi tiếp theo liên quan đến cùng nhóm sản phẩm (thêm bộ lọc giá, hỏi chi tiết, hỏi tồn kho...), "
            f"HÃY TÁI SỬ DỤNG CHÍNH XÁC các bộ lọc ở trên (brand, category, name_contains, mode, keyword). "
            f"Chỉ điều chỉnh max_price/min_price/limit/include_details/need_price_info theo yêu cầu mới. "
            f"KHÔNG đổi mode (lines/rank) trừ khi khách chuyển rõ ràng sang yêu cầu top-N/mua 1 mẫu cụ thể."
        )
        return ctx

    tool_history = state.get("tool_calls_used") or []
    last_search_record = None
    for rec in reversed(tool_history):
        if rec.get("tool") == "product_search":
            last_search_record = rec
            break

    full_messages = [{"role": "system", "content": FULL_MASTER_PROMPT}]
    user_count = 0
    for msg in state["messages"]:
        role = msg.get("role") if isinstance(msg, dict) else getattr(msg, "type", "")
        if role == "user":
            user_count += 1
            if user_count > 1 and last_search_record:
                note = _fmt_search_context(last_search_record)
                if note:
                    full_messages.append({"role": "system", "content": note})
        full_messages.append(msg)

    auth_context = {
        "user_id": state.get("user_id"),
        "user_token": state.get("user_token")
    }

    result = master_agent.invoke(
        messages=full_messages,
        available_tools=available_tools,
        tools_schema=tools_schema,
        auth_context=auth_context
    )

    new_tool_records = result.get("tool_context") or []

    assistant_msg = {
        "role": "assistant",
        "content": result["content"]
    }

    conversation_state = state.get("conversation_state") or {}
    found_new_search = False
    for rec in reversed(new_tool_records):
        if rec.get("tool") == "product_search":
            conversation_state["last_product_search"] = rec
            found_new_search = True
            break
    if not found_new_search and last_search_record:
        conversation_state["last_product_search"] = last_search_record

    return {
        "final_answer": result["content"],
        "messages": [assistant_msg],
        "tool_calls_used": new_tool_records,
        "conversation_state": conversation_state,
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }

In [10]:
builder = StateGraph(AgentState)

builder.add_node("receive_node", receive_node)
builder.add_node("guardrail_node", guardrail_node)
builder.add_node("rejection_node", rejection_node)
builder.add_node("master_node", master_node)

def route_after_guardrail(state: AgentState) -> str:
    """Hàm quyết định Nút tiếp theo dựa vào kết quả của Guardrail"""
    risk = state.get("risk_level", "safe")
    
    if risk == "attack":
        # ============================================================
        # print("⚠️ [ROUTER] Phát hiện ATTACK ➡️ Rẽ nhánh sang rejection_node")
        # ============================================================

        return "rejection_node"
    else:

        # ============================================================
        # print("✅ [ROUTER] An toàn SAFE ➡️ Rẽ nhánh sang master_node")
        # ============================================================
        
        return "master_node"


builder.add_edge(START, "receive_node")
builder.add_edge("receive_node", "guardrail_node")
builder.add_conditional_edges(
    "guardrail_node",
    route_after_guardrail,
    {
        "rejection_node": "rejection_node", 
        "master_node": "master_node"       
    }
)
builder.add_edge("rejection_node", END)
builder.add_edge("master_node", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)
print("🎉 Đã dựng thành công Đồ thị LangGraph Workflow 1!")

🎉 Đã dựng thành công Đồ thị LangGraph Workflow 1!


cho a hỏi mk có ss s26 k e nhỉ, nếu có thì chính sách bảo hành như nào e?

In [11]:
user_token = "eyJhbGciOiJFUzI1NiIsImtpZCI6ImNiZDkwZGZjLTFkMmEtNDE5My1iNzE2LTlkMDgxOGM2MGEyNCIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJodHRwczovL3JpeWNlbm5kc3JscGl2emJjbm1mLnN1cGFiYXNlLmNvL2F1dGgvdjEiLCJzdWIiOiJmZjY0MWYyNi0zYWRhLTQ3ZDAtOWJmMi0xZjRiNzE2NTQwNjQiLCJhdWQiOiJhdXRoZW50aWNhdGVkIiwiZXhwIjoxNzg1NDA2NDcxLCJpYXQiOjE3ODU0MDI4NzEsImVtYWlsIjoidnVnaWFraGFpMjAwNEBnbWFpbC5jb20iLCJwaG9uZSI6IiIsImFwcF9tZXRhZGF0YSI6eyJwcm92aWRlciI6Imdvb2dsZSIsInByb3ZpZGVycyI6WyJnb29nbGUiXX0sInVzZXJfbWV0YWRhdGEiOnsiYWRkcmVzcyI6IiIsImF2YXRhcl91cmwiOiJodHRwczovL2xoMy5nb29nbGV1c2VyY29udGVudC5jb20vYS9BQ2c4b2NKdW9idmozc1ZPZDRsTE1JUVg2d3l4MEtncjJRNFZvZnlVM2pJYVhLN1p4LV9hcmc9czk2LWMiLCJiaXJ0aGRheSI6IjIwMDAtMDItMjAiLCJlbWFpbCI6InZ1Z2lha2hhaTIwMDRAZ21haWwuY29tIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsImZ1bGxfbmFtZSI6IkIyMkRDS0gwNjVfVsWpIEdpYSBLaOG6o2kiLCJpc3MiOiJodHRwczovL2FjY291bnRzLmdvb2dsZS5jb20iLCJuYW1lIjoiQjIyRENLSDA2NV9WxakgR2lhIEto4bqjaSIsInBob25lIjoiIiwicGhvbmVfdmVyaWZpZWQiOmZhbHNlLCJwaWN0dXJlIjoiaHR0cHM6Ly9saDMuZ29vZ2xldXNlcmNvbnRlbnQuY29tL2EvQUNnOG9jSnVvYnZqM3NWT2Q0bExNSVFYNnd5eDBLZ3IyUTRWb2Z5VTNqSWFYSzdaeC1fYXJnPXM5Ni1jIiwicHJvdmlkZXJfaWQiOiIxMDUxNTE0NjExMTM0OTM3ODM1MzUiLCJyb2xlIjoiYWRtaW4iLCJzdWIiOiIxMDUxNTE0NjExMTM0OTM3ODM1MzUifSwicm9sZSI6ImF1dGhlbnRpY2F0ZWQiLCJhYWwiOiJhYWwxIiwiYW1yIjpbeyJtZXRob2QiOiJvYXV0aCIsInRpbWVzdGFtcCI6MTc4NTMyOTYxMX1dLCJzZXNzaW9uX2lkIjoiNDRlOTRmYzYtNzAyMy00Yzg3LTgyY2QtYjI3MDgyZjA2ZWJkIiwiaXNfYW5vbnltb3VzIjpmYWxzZX0.udNcoJiBba-sxCCNR-P-KOXCJhkQTE6bj0b8Sg5EZSsfFy0zvnSX4qGEtLvWUe7ncAaZbda1EDyy8O8IlBitZw"

In [12]:

run_config = {"configurable": {"thread_id": "session_test_notebook_002"}}

# 1. Gọi Đồ thị chạy
res = app.invoke(
    {"user_query": "Em ơi, bên em có mã laptop văn phòng nào dưới 30 triệu không, tư vấn chị vài mẫu với.",
     "user_token": user_token}, 
    config=run_config
)

# 2. IN BÁO CÁO THỐNG KÊ CHI TIẾT TỪ AGENT STATE
print()
print("📊 BÁO CÁO THỐNG KÊ CHI TIẾT (AGENT STATE METRICS)")
print("="*60)
print(f"💬 Câu trả lời (Final Answer) : {res.get('final_answer')}")
print(f"🛡️ Mức độ rủi ro (Risk Level) : {res.get('risk_level')}")
print(f"⏱️ Tổng độ trễ (Total Latency): {res.get('latency', 0):.2f}s")
print(f"📥 Input Tokens               : {res.get('input_tokens', 0)}")
print(f"📤 Output Tokens              : {res.get('output_tokens', 0)}")
print(f"🧮 Tổng Tokens (Total Tokens)  : {res.get('total_tokens', 0)}")

# 3. IN LỊCH SỬ HỘI THOẠI TRONG MEMORY
print("\n📜 LỊCH SỬ HỘI THOẠI (MESSAGES HISTORY):")
for idx, msg in enumerate(res.get("messages", []), 1):
    role = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else "unknown")
    content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else "")
    print(f"  [{idx}] {role.upper()}: {content}")

print("="*60)


Supabase online token verification failed: invalid JWT: unable to parse or verify signature, token has invalid claims: token is expired
🌀 --- LƯỢT 1 (STREAMING) ---
⏱️ [TTFT] Chữ/Tool đầu tiên xuất hiện sau: 1.28s



⏱️ [TURN 1 LATENCY]: 1.28s
📊 [TURN 1 TOKENS]: Input = 4296 | Output = 46 | Subtotal = 4342

🔧 LLM yêu cầu gọi Tool...
   👉 Chạy hàm: product_search({'queries': [{'mode': 'rank', 'max_price': 30000000, 'keyword': 'văn phòng', 'category': 'laptop', 'limit': 3}]})
   📊 Kết quả từ Tool: ["văn phòng"]:
Product: Laptop ASUS VivoBook X415EA-EK2043W | ID: 66282
- ID: 66282
- Brand: ASUS
- Score: 0.0264

Product: Laptop Asus VivoBook Flip 14 TM420UA-EC182W | ID: 44505
- ID: 44505
- Brand: ASUS
- Score: 0.0241

Product: Laptop ASUS Vivobook 14X A1403ZA-LY072W | ID: 50610
- ID: 50610
- Brand: ASUS
- Score: 0.0238
⏱️ [TOOL EXECUTION TIME]: 3.27s
🔄 Gửi kết quả Tool lại cho Gemini suy luận tiếp...

🌀 --- LƯỢT 2 (STREAMING) ---
⏱️ [TTFT] Chữ/Tool đầu tiên xuất hiện sau: 25.01s



⏱️ [TUR

In [13]:
# from src.h_evaluation.benchmark_evaluator import print_table
# # --- Benchmark Evaluator (gọi trong notebook sau khi đã có `app` và `llm_service`) ---
# # Mặc định dùng judge LLM nhanh, không null. Judge mặc định là Groq openai/gpt-oss-120b.
# # Đặt use_ragas=True nếu muốn chạy RAGAS thật (chậm hơn nhiều).
# from src.h_evaluation.benchmark_evaluator import BenchmarkEvaluator

# evaluator = BenchmarkEvaluator(llm_service=llm_service)  # Groq gpt-oss-120b mặc định
# # # evaluator = BenchmarkEvaluator(llm_service=llm_service, judge_provider='gemini',
# # #                                  judge_model=config.llm.google.available[2])  # gemma-4-26b-a4b-it
# # # evaluator = BenchmarkEvaluator(llm_service=llm_service, judge_provider='groq',
# # #                                  judge_model=config.llm.groq.available[1])  # openai/gpt-oss-120b

# test_path = Path(rag_service_dir) / "src" / "h_evaluation" / "test_sets" / "ecommerce_benchmark_20each.jsonl"

# report = evaluator.run_and_evaluate(
#     app,
#     user_token=user_token,
#     benchmark=test_path,
#     output_dir='benchmark_results',
#     max_samples=240,
# )

# print_table("benchmark_results")


In [2]:
from src.h_evaluation.benchmark_evaluator import print_table
from src.h_evaluation.benchmark_evaluator import BenchmarkEvaluator

print_table("benchmark_results")


c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\instructor\providers\gemini\client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


📂 Nạp dữ liệu per-row thô (per_row_scores_1785413164.csv) & tính toán lại chỉ số theo logic mới...

                            📊 AGENTIC RAG BENCHMARK EXECUTIVE DASHBOARD                             
📌 Tổng số mẫu: 302 | ❌ Thất bại: 0 | ⏱️ Latency Trung Bình: 7.87s (p95: 14.47s)

🎯 1. CHẤT LƯỢNG RAGAS & JUDGE EVALUATION (0.0 - 1.0)
┌──────────────────────────────────────────────┬──────────┬──────────┬──────────┬──────────┐
│ Metric                                       │   Mean   │  Median  │ P5(Sàn)  │ % < 0.5  │
├──────────────────────────────────────────────┼──────────┼──────────┼──────────┼──────────┤
│ 🟢 Faithfulness (Độ trung thực)               │  0.7177  │  1.0000  │  0.0000  │   19.2%   │
│ 🟢 Answer Correctness (Độ chính xác)          │  0.5849  │  1.0000  │  0.0000  │   33.8%   │
│ 🟢 Answer Relevancy (Độ liên quan)            │  0.8907  │  1.0000  │  0.5000  │    2.0%   │
│ 🔵 Context Precision (Độ đúng)*               │  0.8546  │  1.0000  │  0.0000  │    6.4%   │
│ 🔵 Contex

Em thưa cô, em là Khải ạ, đang theo thực tập tại trường với sự hưỡng dẫn của cô ạ. Em có 1 số câu hỏi về phần agent + rag với đề tài của em ạ:

- Trước hết, em hiện tại đang thiết kế hệ thống agent theo dạng graph, em đang thiết kế theo 2 nhánh và dùng để benchmark. Nhánh đầu tiên, em giao cho agent toàn quyền quyết định gọi tool, nhánh thứ 2 em tách các tool riêng ra thành các node riêng với từng chức năng.
- Các tool hiện em có là :
    - product_search 
    - product_compare
    - policy_search
    - order_lookup (để tìm kiếm, xem tình trạng đơn hàng của khách hàng ứng với user id riêng và cần xác thực). 
    - Ngoài ra, với các câu hỏi mang rủi ro về mặt kinh tế, agent sẽ khuyên người dùng tự tạo ticken chat với nhân viên

- Các metric đánh giá của em bao gồm:
  - Faithfulness: Độ trung thực
  - Answer Correctness (Độ chính xác)
  - Answer Relevancy (Độ liên quan)
  - Context Precision (Độ đúng ngữ cảnh)
  - Context Recall (Độ phủ ngữ cảnh)
  - Tool Calls Distribution (Phân bố mức gọi Tool), avg tool call
  - Latency (Độ trễ phản hồi - Giây)
  - Token Usage (Lượng Token tiêu thụ) (in/out)



Vấn đề của em:
- Hiện em đã cây dựng xong khung agent react graph, nhưng em chưa biết đánh giá như nào cho hợp lý. Em có dùng RAGAS để sinh test set và đánh giá cho hệ thống. nhưng em nhận thấy ragas chỉ sinh được các câu hỏi liên quan đến hỏi đơn sản phẩm, mà thực tế thì có các usecase như:
    - Câu hỏi về hàng, chính sách gồm: hỏi đơn, hỏi đa sản phẩm, hỏi kết hợp đơn/đa điều kiện (top n sản phẩm, giá, dòng), hội thoại multi-turn,...
    - An toàn và bảo mật: attack, các câu hỏi rủi ro về kinh tế
    - Các câu hỏi mơ hồ
- Em có nhờ ai tư vấn và viết ra 1 file sinh data riêng dựa theo ragas là từ đáp án sinh ngược câu hỏi và em custom 1 file benchmark riêng, gồm các thang đo như ở trên. Nhưng trong quá trình benchmark, với các usecase đơn điểm khá cao, nhưng với nhiều usecase hỏi chi tiết hay multi-turn thì điểm lại thấp mặc dù em check chay log agent thì thấy trả về vẫn đúng nội dung, khôgn sai lệch


- Với bài đo lantency, việc gọi api thi thoảng em để ý phàn hồi khá lâu, em nghi là nghẽn api nên benchmark phần này không phàn ảnh được từ nhiều lần